In [ ]:
#    VAMOS A CREAR UN RAG ARCAICO CON UNA BASE DE INFORMACION PERSISTENTE


#  1) DEFINIMOS EL TEXTO QUE CONFORMA LA BASE DE CONOCIMIENTO
#  2) DEFINIMOS LAS FUNCIONES QUE NECESITAMOS (CHUNKING,CONVERSION DE LOS EMBEDDINGS Y TECNICA DE SIMILITUD,LLAMADA AL LLM)
#  3) CREAMOS UN ARCHIVO TEMPORAL CON LOS CHUNKS OBTENIDOS
#  4) AMPLICAMOS EL ALGORITMO DE EMBEDDINGS A LOS CHUNKS
#  5) CREAMOS UN ARCHIVO JSON CON UNA LISTA DE OBJETOS CON LOS ELEMENTOS  ID,CHUNK,EMBEDDINGS
#  6) EJECUTAMOS UNA CONSULTA Y SU TRANSFORMACIÓN EN EMBEDDING
#  7) REALIZAMOS LA LISTA DE DISTANCIA SEMANTICA Y LA ORDENAMOS DE MAS CERCA A MAS LEJOS (si es mayor o menor dependera de la tecnica usada)
#  8) DEVOLVEMOS EL TOP K ELEMENTOS MAS CERCANOS Y LOS INCRUSTAMOS COMO PARTE DEL CONTEXTO DEL PROMPT
#  9) HACEMOS LA LLAMADA AL PROMPT CON LA PREGUNTA ORIGINAL DEL USUARIO Y EL CONTEXTO DE NUESTRA BASE DE CONOCIMIENTO

In [1]:
!pip install sentence-transformers groq scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00


In [10]:
texto_base = """
Juan Perez tiene la factura 1001 por 1500 pesos.
La factura número 1002 corresponde a Maria Gomez con un total de 2300 pesos.
ACME SA posee la orden 2001 que fue aprobada.
TechCorp mantiene la orden 2002 en estado pendiente.
Global SRL registró la factura 1003 por 1800 pesos.
El cliente Carlos Lopez tiene una factura 1004 por 2100 pesos.
Se generó la orden 2003 para Innovatech, actualmente aprobada.
Factura 1005 emitida a Laura Diaz por un monto de 3200 pesos.
Orden 2004 vinculada a ACME SA sigue en revisión.
Maria Gomez cuenta con la factura 1006 por 2750 pesos.
Juan Perez figura en la factura 1007 con un importe de 4100 pesos.
La orden 2005 del cliente TechCorp fue rechazada.
Global SRL tiene registrada la orden 2006 aprobada.
Carlos Lopez aparece en la factura 1008 con valor 1250 pesos.
Laura Diaz tiene asociada la orden 2007 en estado pendiente.
Factura 1009 emitida para Innovatech por 3600 pesos.
ACME SA registra la factura 1010 por un total de 1900 pesos.
Orden 2008 creada para Maria Gomez y aprobada.
Factura 1011 del cliente Juan Perez por 2400 pesos.
La orden 2009 corresponde a Global SRL y sigue pendiente.
Se emitió la factura 1012 para Laura Diaz con valor de 3000.
Orden 2010 del cliente Innovatech fue rechazada.
ACME SA tiene la factura 1013 por 4100 pesos.
Maria Gomez figura en la orden 2011 aprobada.
Factura 1014 vinculada a Juan Perez por 2200 pesos.
Orden 2012 asignada a Carlos Lopez fue rechazada.
TechCorp cuenta con la factura 1015 por 3300 pesos.
Laura Diaz tiene la orden 2013 aprobada.
Factura 1016 corresponde a Global SRL con 2700 pesos.
ACME SA mantiene la orden 2014 en estado pendiente.
Maria Gomez posee la factura 1017 por 3100 pesos.
Orden 2015 creada para Innovatech fue aprobada.
Carlos Lopez registra la factura 1018 por 2900 pesos.
Juan Perez tiene la orden 2016 en estado pendiente.
Factura 1019 emitida a Laura Diaz por 3500 pesos.
Orden 2017 vinculada a TechCorp fue aprobada.
Factura 1020 correspondiente a Global SRL por 2800 pesos.
Orden 2018 creada para ACME SA sigue pendiente.
Se generó la factura 1021 para Maria Gomez por 2600 pesos.
Orden 2019 del cliente Juan Perez fue aprobada.
Factura 1022 emitida a Carlos Lopez por 2100 pesos.
La orden 2020 para Innovatech sigue en revisión.
Factura 1023 corresponde a TechCorp por 3900 pesos.
Orden 2021 de Global SRL fue aprobada.
Factura 1024 registrada para Laura Diaz por 3100 pesos.
Orden 2022 vinculada a ACME SA fue rechazada.
Factura 1025 emitida a Juan Perez por 1800 pesos.
Orden 2023 del cliente Maria Gomez sigue pendiente.
Factura 1026 corresponde a Carlos Lopez con valor 2000.
Orden 2024 para TechCorp fue aprobada.
Factura 1027 emitida a Innovatech por 4500 pesos.
Orden 2025 creada para Global SRL fue rechazada.
Factura 1028 corresponde a ACME SA por 3300 pesos.
Orden 2026 del cliente Laura Diaz sigue pendiente.
Factura 1029 emitida a Juan Perez por 1500 pesos.
Orden 2027 de Carlos Lopez fue aprobada.
Factura 1030 registrada para Maria Gomez por 2700 pesos.
Orden 2028 vinculada a TechCorp fue rechazada.
Factura 1031 emitida a Global SRL por 2900 pesos.
Orden 2029 para Innovatech sigue pendiente.
Factura 1032 correspondiente a ACME SA por 3600 pesos.
Orden 2030 del cliente Juan Perez fue aprobada.
Factura 1033 emitida a Laura Diaz por 3100 pesos.
Orden 2031 de Maria Gomez fue rechazada.
Factura 1034 registrada para Carlos Lopez por 2800 pesos.
Orden 2032 vinculada a TechCorp sigue pendiente.
Factura 1035 emitida a Innovatech por 4200 pesos.
Orden 2033 del cliente Global SRL fue aprobada.
Factura 1036 corresponde a ACME SA por 3000 pesos.
Orden 2034 para Laura Diaz fue rechazada.
Factura 1037 emitida a Juan Perez por 2600 pesos.
Orden 2035 de Carlos Lopez sigue pendiente.
Factura 1038 registrada para Maria Gomez por 2200 pesos.
Orden 2036 vinculada a TechCorp fue aprobada.
Factura 1039 emitida a Global SRL por 3700 pesos.
Orden 2037 del cliente Innovatech fue rechazada.
Factura 1040 corresponde a ACME SA por 2800 pesos.
Orden 2038 para Juan Perez sigue pendiente.
Factura 1041 emitida a Laura Diaz por 3300 pesos.
Orden 2039 de Maria Gomez fue aprobada.
Factura 1042 registrada para Carlos Lopez por 3100 pesos.
Orden 2040 vinculada a TechCorp fue rechazada.
Factura 1043 emitida a Innovatech por 3900 pesos.
Orden 2041 del cliente Global SRL sigue pendiente.
Factura 1044 corresponde a ACME SA por 3500 pesos.
Orden 2042 para Laura Diaz fue aprobada.
Factura 1045 emitida a Juan Perez por 2700 pesos.
Orden 2043 de Carlos Lopez fue rechazada.
Factura 1046 registrada para Maria Gomez por 2400 pesos.
Orden 2044 vinculada a TechCorp sigue pendiente.
Factura 1047 emitida a Global SRL por 3100 pesos.
Orden 2045 del cliente Innovatech fue aprobada.
Factura 1048 corresponde a ACME SA por 3600 pesos.
Orden 2046 para Juan Perez fue rechazada.
Factura 1049 emitida a Laura Diaz por 2900 pesos.
Orden 2047 de Maria Gomez sigue pendiente.
Factura 1050 registrada para Carlos Lopez por 2800 pesos.
"""

In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json

#Modelo de embeddings

model = SentenceTransformer("all-MiniLM-L6-v2")

#Chunking simple por oraciones

def chunking(texto):
  return [line.strip() for line in texto.split("\n") if line.strip()]

#Generar embeddings

def generar_embeddings(textos):
  return model.encode(textos)

#Búsqueda por similitud coseno

def buscar_similares(query_embedding, embeddings, top_k=3):
  scores = cosine_similarity([query_embedding], embeddings)[0]
  indices = np.argsort(scores)[::-1]
  return indices, scores

#Llamada al LLM (Groq)

from groq import Groq
from google.colab import userdata

api_groq = userdata.get('GROQ_API_KEY')

client = Groq(api_key=api_groq)

def consultar_llm(prompt):
  response = client.chat.completions.create(
  model="llama-3.1-8b-instant",
  messages=[{"role": "user", "content": prompt}]
  )
  return response.choices[0].message.content

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
#CREAMOS ARCHIVO TEMPORAL CON LOS CHUNKS


chunks = chunking(texto_base)

with open("temporal.txt", "w") as f:
  for chunk in chunks:
    f.write(chunk + "\n")

#--------------------------------------------------
#APLICAMOS EMBEDDINGS A LOS CHUNKS
#--------------------------------------------------

embeddings = generar_embeddings(chunks)


# CREAMOS JSON CON ID, CHUNK Y EMBEDDING

base = []

for i, chunk in enumerate(chunks):
  base.append({
  "id": i,
  "chunk": chunk,
  "embedding": embeddings[i].tolist()
  })

with open("base.json", "w") as f:
  json.dump(base, f)




In [22]:
#CONSULTA DEL USUARIO + EMBEDDING


query = "¿Qué factura tiene Juan Perez?"



query_embedding = model.encode(query)


# CALCULO DE SIMILITUD Y ORDENAMIENTO

embeddings_array = np.array([item["embedding"] for item in base])

indices, scores = buscar_similares(query_embedding, embeddings_array)


# TOP K RESULTADOS → CONTEXTO

# Ordenamos de mayor a menor
indices_ordenados = np.argsort(scores)[::-1]

# Tomamos top K
top_k = 5
indices = indices_ordenados[:top_k]


top_chunks = [base[i]["chunk"] for i in indices]

contexto = "\n".join(top_chunks)




In [23]:
prompt = f"""
Responde la pregunta usando únicamente el siguiente contexto.

Contexto:
{contexto}

Pregunta:
{query}
"""

respuesta = consultar_llm(prompt)

In [24]:
print("=== CONTEXTO UTILIZADO ===")
print(contexto)

print("\n=== RESPUESTA DEL MODELO ===")
print(respuesta)

=== CONTEXTO UTILIZADO ===
Juan Perez tiene la factura 1001 por 1500 pesos.
Factura 1014 vinculada a Juan Perez por 2200 pesos.
Factura 1029 emitida a Juan Perez por 1500 pesos.
Factura 1045 emitida a Juan Perez por 2700 pesos.
Factura 1011 del cliente Juan Perez por 2400 pesos.

=== RESPUESTA DEL MODELO ===
Según el contexto proporcionado, Juan Perez tiene las siguientes facturas:

1. Factura 1001 por 1500 pesos
2. Factura 1011 por 2400 pesos
3. Factura 1014 por 2200 pesos
4. Factura 1029 por 1500 pesos
5. Factura 1045 por 2700 pesos
